In [1]:

# ================================================================
# CIFAR-10 RESNET ARCHITECTURE CHALLENGE
# COMPLETE SINGLE-CELL KAGGLE SOLUTION
#
# Fixes image filename matching
# Trains from scratch on official competition data
# Single model, under 5 million parameters
# Generates /kaggle/working/submission.csv
# ================================================================

import os
import re
import math
import time
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

warnings.filterwarnings("ignore")

# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42
EPOCHS = 220
BATCH_SIZE = 128
NUM_WORKERS = 2

LEARNING_RATE = 0.1
WEIGHT_DECAY = 5e-4
WARMUP_EPOCHS = 5

EMA_DECAY = 0.999
MIX_PROBABILITY = 0.5
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 0.5
CLEAN_LAST_EPOCHS = 20

# False = train on all 50,000 official images.
# True = use 45,000 training and 5,000 validation.
USE_VALIDATION = False

# Horizontal flip test-time augmentation.
USE_TTA = True

# Resume from recovery checkpoint when available.
RESUME = True
SAVE_EVERY = 10

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if DEVICE.type != "cuda":
    raise RuntimeError(
        "GPU not enabled. Open Kaggle Settings "
        "and select GPU T4 or P100."
    )

torch.backends.cudnn.benchmark = True

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("=" * 65)
print("CIFAR-10 RESNET CHALLENGE")
print("GPU:", torch.cuda.get_device_name(0))
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("=" * 65)

# ================================================================
# 2. FIND DATASET
# ================================================================

def find_dataset_root():
    candidates = list(
        Path("/kaggle/input").rglob("train_labels.csv")
    )

    for csv_path in candidates:
        root = csv_path.parent

        if (
            (root / "train_images").is_dir()
            and (root / "test_images").is_dir()
            and (root / "sample_submission.csv").exists()
        ):
            return root

    raise FileNotFoundError(
        "Official competition dataset not found. "
        "Attach it using Kaggle Add Input."
    )

ROOT = find_dataset_root()

train_df = pd.read_csv(
    ROOT / "train_labels.csv"
)

sample_df = pd.read_csv(
    ROOT / "sample_submission.csv"
)

assert {"id", "label"}.issubset(train_df.columns)
assert {"id", "label"}.issubset(sample_df.columns)

assert train_df["id"].is_unique
assert sample_df["id"].is_unique

raw_labels = train_df["label"].to_numpy()

classes = sorted(
    pd.unique(raw_labels).tolist()
)

assert len(classes) == 10, (
    f"Expected 10 classes, got {classes}"
)

class_to_index = {
    value: index
    for index, value in enumerate(classes)
}

index_to_class = {
    index: value
    for value, index in class_to_index.items()
}

labels = np.array(
    [class_to_index[x] for x in raw_labels],
    dtype=np.int64
)

print("\nDataset:", ROOT)
print("Training rows:", len(train_df))
print("Test rows:", len(sample_df))
print("Class mapping:", class_to_index)

# ================================================================
# 3. ROBUST IMAGE MATCHING
# ================================================================

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".webp", ".bmp"
}

def match_images(folder, ids):
    folder = Path(folder)

    files = sorted(
        [
            p for p in folder.rglob("*")
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTENSIONS
        ],
        key=lambda p: str(p)
    )

    if not files:
        raise FileNotFoundError(
            f"No images found inside {folder}"
        )

    print(f"\nScanning: {folder}")
    print("Images discovered:", len(files))
    print("Example filenames:")
    for p in files[:10]:
        print(" ", p.relative_to(folder))

    by_relative = {}
    by_name = {}
    by_stem = {}
    by_number = {}

    def add_lookup(mapping, key, path):
        if key not in mapping:
            mapping[key] = path
        elif mapping[key] != path:
            mapping[key] = None

    for path in files:
        relative = path.relative_to(folder).as_posix().lower()
        name = path.name.lower()
        stem = path.stem.lower()

        add_lookup(by_relative, relative, path)
        add_lookup(by_name, name, path)
        add_lookup(by_stem, stem, path)

        # Supports names such as:
        # 0.png, 00000.png, img_0.png, image_0.png
        numbers = re.findall(r"\d+", stem)

        if len(numbers) == 1:
            number = int(numbers[0])
            add_lookup(by_number, number, path)

    matched = []

    for image_id in ids:
        key = str(image_id).strip().lower()

        candidate_keys = [
            key,
            Path(key).stem
        ]

        found = None

        # Prefer exact relative path / filename / stem.
        for candidate in candidate_keys:
            if candidate in by_relative:
                found = by_relative[candidate]
                if found is None:
                    raise ValueError(
                        f"Ambiguous relative path: {candidate}"
                    )
                break

            if candidate in by_name:
                found = by_name[candidate]
                if found is None:
                    raise ValueError(
                        f"Ambiguous filename: {candidate}"
                    )
                break

            if candidate in by_stem:
                found = by_stem[candidate]
                if found is None:
                    raise ValueError(
                        f"Ambiguous filename stem: {candidate}"
                    )
                break

        # Fallback: numeric image ID.
        if found is None:
            try:
                number = int(image_id)

                if number in by_number:
                    found = by_number[number]

                    if found is None:
                        raise ValueError(
                            f"Ambiguous numeric ID: {image_id}"
                        )
            except (TypeError, ValueError) as exc:
                if "Ambiguous numeric ID" in str(exc):
                    raise

        if found is None:
            raise FileNotFoundError(
                f"Cannot match image ID {image_id} "
                f"inside {folder}. "
                f"Examples: {[p.name for p in files[:10]]}"
            )

        matched.append(found)

    if len(set(matched)) != len(matched):
        raise ValueError(
            "Different IDs mapped to the same image. "
            "Stopping to prevent an invalid submission."
        )

    print("Successfully matched:", len(matched))
    print("First five ID-to-image mappings:")

    for image_id, path in list(zip(ids, matched))[:5]:
        print(
            f"  ID {image_id} -> "
            f"{path.relative_to(folder)}"
        )

    return matched

train_paths = match_images(
    ROOT / "train_images",
    train_df["id"].tolist()
)

test_paths = match_images(
    ROOT / "test_images",
    sample_df["id"].tolist()
)

assert len(train_paths) == 50000
assert len(test_paths) == 2000

# Check image dimensions and format.
for path in train_paths[:5] + test_paths[:5]:
    with Image.open(path) as image:
        assert image.size == (32, 32), (
            f"Unexpected image size: {path}: {image.size}"
        )

print("\nImage matching completed.")

# ================================================================
# 4. DATA AUGMENTATION
# ================================================================

train_transform = T.Compose([
    T.RandomCrop(
        32,
        padding=4,
        padding_mode="reflect"
    ),
    T.RandomHorizontalFlip(),
    T.RandAugment(
        num_ops=2,
        magnitude=5
    ),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    T.RandomErasing(
        p=0.10,
        scale=(0.02, 0.08),
        ratio=(0.5, 2.0),
        value=0
    )
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

class CIFARDataset(Dataset):
    def __init__(
        self,
        paths,
        targets=None,
        transform=None
    ):
        self.paths = paths
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert("RGB")

        image = self.transform(image)

        if self.targets is None:
            return image

        return image, int(self.targets[index])

def create_loader(
    paths,
    targets,
    transform,
    shuffle
):
    dataset = CIFARDataset(
        paths,
        targets,
        transform
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0
    )

# ================================================================
# 5. TRAIN / VALIDATION SPLIT
# ================================================================

if USE_VALIDATION:
    rng = np.random.default_rng(SEED)

    train_indices = []
    validation_indices = []

    for cls in range(10):
        indices = np.where(
            labels == cls
        )[0]

        rng.shuffle(indices)

        n_validation = int(
            round(len(indices) * 0.10)
        )

        validation_indices.extend(
            indices[:n_validation]
        )

        train_indices.extend(
            indices[n_validation:]
        )

    train_indices = np.array(
        train_indices,
        dtype=np.int64
    )

    validation_indices = np.array(
        validation_indices,
        dtype=np.int64
    )
else:
    train_indices = np.arange(
        len(train_paths)
    )
    validation_indices = None

train_loader = create_loader(
    [train_paths[i] for i in train_indices],
    labels[train_indices],
    train_transform,
    shuffle=True
)

validation_loader = None

if validation_indices is not None:
    validation_loader = create_loader(
        [train_paths[i] for i in validation_indices],
        labels[validation_indices],
        eval_transform,
        shuffle=False
    )

test_loader = create_loader(
    test_paths,
    None,
    eval_transform,
    shuffle=False
)

print("\nTraining samples:", len(train_indices))

if validation_indices is not None:
    print(
        "Validation samples:",
        len(validation_indices)
    )

print("Test samples:", len(test_paths))

# ================================================================
# 6. SQUEEZE-AND-EXCITATION
# ================================================================

class SEBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        hidden = max(
            8,
            channels // 16
        )

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1
        )

        self.fc2 = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1
        )

    def forward(self, x):
        scale = F.adaptive_avg_pool2d(
            x, 1
        )

        scale = F.relu(
            self.fc1(scale),
            inplace=True
        )

        scale = torch.sigmoid(
            self.fc2(scale)
        )

        return x * scale

# ================================================================
# 7. RESIDUAL BLOCK
# ================================================================

class ResidualBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(
            out_channels
        )

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(
            out_channels
        )

        self.se = SEBlock(
            out_channels
        )

        if (
            stride != 1
            or in_channels != out_channels
        ):
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(
                    out_channels
                )
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        x = F.relu(
            self.bn1(self.conv1(x)),
            inplace=True
        )

        x = self.bn2(self.conv2(x))
        x = self.se(x)

        return F.relu(
            x + identity,
            inplace=True
        )

# ================================================================
# 8. EFFICIENT RESNET
# ================================================================

class EfficientResNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(
                3, 64,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.stage1 = self.make_stage(
            64, 64, 3, 1
        )

        self.stage2 = self.make_stage(
            64, 128, 3, 2
        )

        self.stage3 = self.make_stage(
            128, 256, 3, 2
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, 10)

        self.apply(self.initialize)

    @staticmethod
    def initialize(module):
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(
                module.weight,
                mode="fan_out",
                nonlinearity="relu"
            )

        elif isinstance(
            module, nn.BatchNorm2d
        ):
            nn.init.ones_(
                module.weight
            )
            nn.init.zeros_(
                module.bias
            )

    def make_stage(
        self,
        cin,
        cout,
        blocks,
        stride
    ):
        layers = [
            ResidualBlock(
                cin,
                cout,
                stride
            )
        ]

        for _ in range(1, blocks):
            layers.append(
                ResidualBlock(
                    cout,
                    cout
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)

        return self.fc(x)

model = EfficientResNet().to(DEVICE)

PARAMETER_COUNT = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "\nTrainable parameters:",
    f"{PARAMETER_COUNT:,}"
)

assert PARAMETER_COUNT <= 5_000_000, (
    "Model exceeds the 5-million-parameter limit."
)

print("Parameter constraint: PASSED")

# ================================================================
# 9. MIXUP AND CUTMIX
# ================================================================

def mixup_cutmix(images, targets):
    if random.random() > MIX_PROBABILITY:
        return (
            images,
            targets,
            targets,
            1.0
        )

    indices = torch.randperm(
        images.size(0),
        device=images.device
    )

    targets_b = targets[indices]

    if random.random() < 0.5:
        lam = float(
            np.random.beta(
                MIXUP_ALPHA,
                MIXUP_ALPHA
            )
        )

        mixed = (
            lam * images
            + (1 - lam) * images[indices]
        )

        return (
            mixed,
            targets,
            targets_b,
            lam
        )

    lam = float(
        np.random.beta(
            CUTMIX_ALPHA,
            CUTMIX_ALPHA
        )
    )

    _, _, height, width = images.shape

    ratio = math.sqrt(
        1.0 - lam
    )

    cut_width = int(
        width * ratio
    )

    cut_height = int(
        height * ratio
    )

    cx = random.randrange(width)
    cy = random.randrange(height)

    x1 = max(
        0,
        cx - cut_width // 2
    )

    x2 = min(
        width,
        cx + cut_width // 2
    )

    y1 = max(
        0,
        cy - cut_height // 2
    )

    y2 = min(
        height,
        cy + cut_height // 2
    )

    mixed = images.clone()

    mixed[:, :, y1:y2, x1:x2] = (
        images[
            indices,
            :,
            y1:y2,
            x1:x2
        ]
    )

    lam = 1.0 - (
        (x2 - x1) * (y2 - y1)
        / (width * height)
    )

    return (
        mixed,
        targets,
        targets_b,
        lam
    )

# ================================================================
# 10. EMA
# ================================================================

ema_model = copy.deepcopy(
    model
).eval()

for parameter in ema_model.parameters():
    parameter.requires_grad_(False)

@torch.no_grad()
def update_ema(
    ema,
    source,
    decay
):
    for ema_parameter, parameter in zip(
        ema.parameters(),
        source.parameters()
    ):
        ema_parameter.mul_(decay).add_(
            parameter.detach(),
            alpha=1.0 - decay
        )

    # Copy BN buffers from the current model.
    for ema_buffer, buffer in zip(
        ema.buffers(),
        source.buffers()
    ):
        ema_buffer.copy_(buffer)

# ================================================================
# 11. OPTIMIZER AND SCHEDULER
# ================================================================

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=0.9,
    weight_decay=WEIGHT_DECAY,
    nesterov=True
)

total_steps = (
    EPOCHS * len(train_loader)
)

warmup_steps = (
    WARMUP_EPOCHS * len(train_loader)
)

def lr_multiplier(step):
    if step < warmup_steps:
        return (
            0.01
            + 0.99
            * (step + 1)
            / max(1, warmup_steps)
        )

    progress = (
        step - warmup_steps
    ) / max(
        1,
        total_steps - warmup_steps
    )

    progress = min(
        1.0,
        max(0.0, progress)
    )

    return (
        0.001
        + 0.999
        * 0.5
        * (
            1.0
            + math.cos(
                math.pi * progress
            )
        )
    )

scheduler = (
    torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lr_multiplier
    )
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

# ================================================================
# 12. VALIDATION / INFERENCE
# ================================================================

@torch.inference_mode()
def predict_probabilities(
    network,
    images,
    use_tta=True
):
    network.eval()

    logits = network(images)

    if use_tta:
        flipped_logits = network(
            images.flip(-1)
        )

        logits = (
            logits + flipped_logits
        ) / 2.0

    return logits.softmax(dim=1)

@torch.inference_mode()
def evaluate(network, loader):
    network.eval()

    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True
        )

        probabilities = predict_probabilities(
            network,
            images,
            use_tta=USE_TTA
        )

        predictions = probabilities.argmax(
            dim=1
        )

        correct += (
            predictions == targets
        ).sum().item()

        total += targets.size(0)

    return correct / total

# ================================================================
# 13. CHECKPOINT RECOVERY
# ================================================================

recovery_path = (
    OUTPUT_DIR / "training_recovery.pth"
)

best_path = (
    OUTPUT_DIR / "best_model.pth"
)

final_path = (
    OUTPUT_DIR / "final_model.pth"
)

start_epoch = 0
global_step = 0
best_accuracy = -1.0
history = []

if RESUME and recovery_path.exists():
    print("\nRecovery checkpoint found.")

    checkpoint = torch.load(
        recovery_path,
        map_location=DEVICE,
        weights_only=False
    )

    # Only resume matching training configurations.
    saved_config = checkpoint.get(
        "config", {}
    )

    current_config = {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "use_validation": USE_VALIDATION,
        "training_samples": len(train_indices),
        "parameter_count": PARAMETER_COUNT
    }

    if saved_config == current_config:
        model.load_state_dict(
            checkpoint["model"]
        )

        ema_model.load_state_dict(
            checkpoint["ema"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer"]
        )

        scheduler.load_state_dict(
            checkpoint["scheduler"]
        )

        scaler.load_state_dict(
            checkpoint["scaler"]
        )

        start_epoch = checkpoint["epoch"]
        global_step = checkpoint["global_step"]
        best_accuracy = checkpoint["best_accuracy"]
        history = checkpoint.get("history", [])

        print(
            "Resuming from epoch:",
            start_epoch
        )
    else:
        print(
            "Checkpoint configuration differs. "
            "Starting from scratch."
        )

# ================================================================
# 14. TRAIN MODEL
# ================================================================

print("\n" + "=" * 65)
print("STARTING TRAINING")
print("=" * 65)

start_time = time.time()

for epoch in range(
    start_epoch,
    EPOCHS
):
    model.train()

    running_loss = 0.0
    processed = 0

    for images, targets in train_loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True
        )

        if (
            epoch
            < EPOCHS - CLEAN_LAST_EPOCHS
        ):
            images, ya, yb, lam = (
                mixup_cutmix(
                    images,
                    targets
                )
            )
        else:
            ya = targets
            yb = targets
            lam = 1.0

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(images)

            loss_a = F.cross_entropy(
                logits,
                ya,
                label_smoothing=0.05
            )

            loss_b = F.cross_entropy(
                logits,
                yb,
                label_smoothing=0.05
            )

            loss = (
                lam * loss_a
                + (1.0 - lam) * loss_b
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()
        scheduler.step()

        global_step += 1

        decay = min(
            EMA_DECAY,
            (global_step + 1)
            / (global_step + 10)
        )

        update_ema(
            ema_model,
            model,
            decay
        )

        running_loss += (
            loss.item()
            * images.size(0)
        )

        processed += (
            images.size(0)
        )

    train_loss = (
        running_loss / processed
    )

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "learning_rate": (
            optimizer.param_groups[0]["lr"]
        )
    }

    if validation_loader is not None:
        val_accuracy = evaluate(
            ema_model,
            validation_loader
        )

        row["val_accuracy"] = val_accuracy

        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy

            torch.save(
                ema_model.state_dict(),
                best_path
            )

        print(
            f"Epoch {epoch+1:03d}/{EPOCHS} | "
            f"Loss: {train_loss:.4f} | "
            f"Val: {val_accuracy*100:.2f}% | "
            f"Best: {best_accuracy*100:.2f}%"
        )

    else:
        print(
            f"Epoch {epoch+1:03d}/{EPOCHS} | "
            f"Loss: {train_loss:.4f}"
        )

    history.append(row)

    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "training_history.csv",
        index=False
    )

    # Save recovery checkpoint.
    if (
        (epoch + 1) % SAVE_EVERY == 0
        or epoch + 1 == EPOCHS
    ):
        checkpoint = {
            "epoch": epoch + 1,
            "global_step": global_step,
            "model": model.state_dict(),
            "ema": ema_model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "best_accuracy": best_accuracy,
            "history": history,
            "config": {
                "epochs": EPOCHS,
                "batch_size": BATCH_SIZE,
                "seed": SEED,
                "use_validation": USE_VALIDATION,
                "training_samples": len(train_indices),
                "parameter_count": PARAMETER_COUNT
            }
        }

        torch.save(
            checkpoint,
            recovery_path
        )

print("\nTraining completed.")

print(
    "Elapsed minutes:",
    round(
        (time.time() - start_time) / 60,
        2
    )
)

# ================================================================
# 15. SELECT MODEL FOR INFERENCE
# ================================================================

if (
    validation_loader is not None
    and best_path.exists()
):
    ema_model.load_state_dict(
        torch.load(
            best_path,
            map_location=DEVICE,
            weights_only=True
        )
    )

    print(
        "Best validation accuracy:",
        f"{best_accuracy*100:.2f}%"
    )

torch.save(
    ema_model.state_dict(),
    final_path
)

ema_model.eval()

# ================================================================
# 16. GENERATE TEST PREDICTIONS
# ================================================================

predictions = []

print("\nGenerating predictions...")

with torch.inference_mode():
    for images in test_loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        probabilities = predict_probabilities(
            ema_model,
            images,
            use_tta=USE_TTA
        )

        predicted_indices = (
            probabilities.argmax(
                dim=1
            )
        )

        predictions.extend(
            predicted_indices.cpu().tolist()
        )

official_labels = [
    index_to_class[int(index)]
    for index in predictions
]

# ================================================================
# 17. SAVE KAGGLE SUBMISSION
# ================================================================

submission = pd.DataFrame({
    "id": sample_df["id"].to_numpy(),
    "label": official_labels
})

assert len(submission) == 2000
assert list(submission.columns) == ["id", "label"]
assert submission["id"].equals(
    sample_df["id"]
)
assert submission["id"].is_unique
assert not submission.isna().any().any()
assert set(submission["label"]).issubset(
    set(classes)
)

submission_file = (
    OUTPUT_DIR / "submission.csv"
)

submission.to_csv(
    submission_file,
    index=False
)

print("\n" + "=" * 65)
print("SUBMISSION CREATED SUCCESSFULLY")
print("=" * 65)

print("Submission:", submission_file)
print("Total predictions:", len(submission))
print("Model parameters:", PARAMETER_COUNT)
print("Parameter limit: PASSED")
print("TTA enabled:", USE_TTA)

print("\nFirst 10 predictions:")
print(
    submission.head(10).to_string(
        index=False
    )
)

print("\nPrediction distribution:")
print(
    submission["label"]
    .value_counts()
    .sort_index()
)

print("\nSaved files:")
for path in sorted(
    OUTPUT_DIR.iterdir()
):
    if (
        path.is_file()
        and path.suffix in (".csv", ".pth")
    ):
        print(" -", path.name)

print("\nDONE!")
print(
    "Download /kaggle/working/submission.csv "
    "and upload it to the Kaggle competition."
)


CIFAR-10 RESNET CHALLENGE
GPU: Tesla T4
Epochs: 220
Batch size: 128

Dataset: /kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl
Training rows: 50000
Test rows: 2000
Class mapping: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9}

Scanning: /kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl/train_images
Images discovered: 50000
Example filenames:
  train_00000.png
  train_00001.png
  train_00002.png
  train_00003.png
  train_00004.png
  train_00005.png
  train_00006.png
  train_00007.png
  train_00008.png
  train_00009.png
Successfully matched: 50000
First five ID-to-image mappings:
  ID 0 -> train_00000.png
  ID 1 -> train_00001.png
  ID 2 -> train_00002.png
  ID 3 -> train_00003.png
  ID 4 -> train_00004.png

Scanning: /kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl/test_images
Images discovered: 2000
Example filenames:
  test_0000.png
  test_0001.png
  test_0002.png
  test_0003.png
  test_0004.png
  test_0005.pn